<a href="https://colab.research.google.com/github/ademiiskak126-ui/Diabetes_Prediction/blob/main/SmartCourse_Recommender.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import random

# Названия курсов
course_names = [
    "Python for Beginners", "Web Development Bootcamp", "Machine Learning A-Z",
    "Data Science with Python", "Frontend Developer Course", "AI for Everyone",
    "Django Fullstack", "React Basics", "Advanced Python", "JavaScript Essentials",
    "Data Analysis with R", "Deep Learning Specialization", "Cybersecurity Fundamentals",
    "Cloud Computing Basics", "SQL for Data Science", "UX/UI Design Basics",
    "Kotlin for Android", "iOS App Development", "TensorFlow for Beginners",
    "Node.js Developer", "Angular Basics", "Git & GitHub Mastery", "Flask Web Apps",
    "Blockchain Basics", "Big Data Analytics", "Excel for Data Science", "Java Fundamentals",
    "C++ Programming", "Front-End Projects", "Back-End Development", "Docker Essentials",
    "Kubernetes Basics", "React Native Mobile Apps", "Agile Project Management",
    "AI Ethics", "Data Visualization with Tableau", "Python for Finance",
    "Computer Networks", "Linux Essentials", "Programming in Go", "Scala for Big Data",
    "RPA with UiPath", "Swift for iOS", "Machine Learning with Python",
    "Java Spring Boot", "PHP for Beginners", "Ruby on Rails Basics", "MongoDB Essentials",
    "Cybersecurity Advanced"
]

platforms = ["Coursera", "Udemy", "Stepik", "Codecademy", "FreeCodeCamp"]
languages = ["Python", "JavaScript", "Java", "C++", "R", "Kotlin", "Swift", "General"]
levels = ["Beginner", "Intermediate", "Advanced"]
formats = ["Online", "Offline"]
categories = ["Web", "AI", "Mobile", "Data Science", "Cybersecurity", "Cloud", "General"]

# Генерация данных
data = []
for name in course_names:
    platform = random.choice(platforms)
    language = random.choice(languages)
    level = random.choice(levels)
    category = random.choice(categories)
    format_course = random.choice(formats)
    duration_weeks = random.randint(4, 16)
    duration_hours = duration_weeks * random.randint(2, 6)
    price = random.choice([0, 10, 15, 20, 25, 30, 35, 50])
    rating = round(random.uniform(4.0, 5.0), 1)
    data.append([name, platform, language, level, category, format_course, duration_weeks, duration_hours, price, rating])

# Создание DataFrame
df = pd.DataFrame(data, columns=[
    "Course Name", "Platform", "Language", "Level", "Category", "Format",
    "Duration (weeks)", "Duration (hours)", "Price ($)", "Rating"
])

# Сохраняем CSV
df.to_csv("it_courses_dataset.csv", index=False)
df.head(10)

,Course Name,Platform,Language,Level,Category,Format,Duration (weeks),Duration (hours),Price ($),Rating
0,Python for Beginners,Stepik,JavaScript,Intermediate,Cloud,Offline,11,33,35,4.2
1,Web Development Bootcamp,Udemy,Python,Intermediate,Cybersecurity,Offline,11,33,15,4.4
2,Machine Learning A-Z,Codecademy,JavaScript,Intermediate,AI,Online,9,45,25,5.0
3,Data Science with Python,Coursera,Swift,Advanced,General,Offline,10,60,20,4.1
4,Frontend Developer Course,Codecademy,R,Advanced,Web,Online,16,96,20,4.5
5,AI for Everyone,Udemy,R,Intermediate,General,Online,14,28,25,4.8
6,Django Fullstack,FreeCodeCamp,Swift,Advanced,Mobile,Online,4,8,25,4.0
7,React Basics,Coursera,Python,Beginner,Data Science,Offline,8,24,20,5.0
8,Advanced Python,Codecademy,Python,Beginner,Cybersecurity,Online,14,42,15,4.9
9,JavaScript Essentials,FreeCodeCamp,General,Beginner,AI,Offline,7,14,10,4.0


In [2]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd

def recommend_courses(user_pref, max_price=50, max_duration=50, top_n=5):
    features = ["Language", "Level", "Category", "Format"]

    # Фильтруем по цене и длительности
    filtered_df = df[(df["Price ($)"] <= max_price) & (df["Duration (hours)"] <= max_duration)].copy()

    if filtered_df.empty:
        return "No courses found matching your price/duration preferences!"

    # One-hot кодирование с безопасной обработкой неизвестных категорий
    encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    X = encoder.fit_transform(filtered_df[features])

    # Преобразуем user_pref в DataFrame
    user_df = pd.DataFrame([user_pref])
    user_vector = encoder.transform(user_df)

    # Косинусное сходство
    similarity = cosine_similarity(X, user_vector)

    # Топ-N курсов
    top_indices = np.argsort(similarity.ravel())[::-1][:top_n]
    return filtered_df.iloc[top_indices]

# Пример использования
user_pref = {"Language": "Python", "Level": "Beginner", "Category": "AI", "Format": "Online"}
recommend_courses(user_pref, max_price=30, max_duration=80, top_n=5)

,Course Name,Platform,Language,Level,Category,Format,Duration (weeks),Duration (hours),Price ($),Rating
8,Advanced Python,Codecademy,Python,Beginner,Cybersecurity,Online,14,42,15,4.9
22,Flask Web Apps,Coursera,Python,Advanced,Data Science,Online,11,22,25,5.0
46,Ruby on Rails Basics,Codecademy,Python,Advanced,General,Online,8,32,25,4.9
18,TensorFlow for Beginners,Codecademy,Swift,Beginner,AI,Offline,8,32,25,4.1
12,Cybersecurity Fundamentals,FreeCodeCamp,C++,Beginner,General,Online,15,30,30,4.0


In [3]:
!pip install gradio -q

import gradio as gr

def recommend_interface(language, level, category, format_course, max_price, max_duration):
    user_pref = {"Language": language, "Level": level, "Category": category, "Format": format_course}
    recs = recommend_courses(user_pref, max_price=max_price, max_duration=max_duration, top_n=5)
    return recs[["Course Name", "Platform", "Language", "Level", "Category", "Price ($)", "Rating"]]

# Списки для выпадающих меню
languages_list = df["Language"].unique().tolist()
levels_list = df["Level"].unique().tolist()
categories_list = df["Category"].unique().tolist()
formats_list = df["Format"].unique().tolist()

# Создаём интерфейс
iface = gr.Interface(
    fn=recommend_interface,
    inputs=[
        gr.Dropdown(languages_list, label="Language"),
        gr.Dropdown(levels_list, label="Level"),
        gr.Dropdown(categories_list, label="Category"),
        gr.Dropdown(formats_list, label="Format"),
        gr.Slider(0, 100, value=50, step=5, label="Max Price ($)"),
        gr.Slider(0, 200, value=80, step=5, label="Max Duration (hours)")
    ],
    outputs="dataframe",
    title="SmartCourse Recommender",
    description="Select your preferences to get top IT course recommendations."
)

iface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4e7e7e523461f4ffff.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Project Name: SmartCourse Recommender
Description:
This project is a personalized IT course recommendation system. It allows users to select their preferences—such as programming language, skill level, course category, format (online/offline), maximum price, and duration—and receive the top 5 course suggestions that best match their criteria.
Key Features:
Content-based filtering: Uses user preferences and course features to calculate similarity with cosine similarity.
Interactive interface: Built with Gradio for easy selection of preferences and real-time recommendations.
Flexible dataset: Includes 50+ IT courses with diverse categories like Web, AI, Mobile, Data Science, Cybersecurity, Cloud, and more.
Filters: Users can filter by price and duration to find courses that fit their schedule and budget.
Dynamic recommendations: Works even if the user selects preferences not present in the dataset (handled safely with OneHotEncoder).
Example Output:
Course Name
Platform
Language
Level
Category
Price ($)
Rating
Machine Learning A-Z
Udemy
Python
Beginner
AI
25
4.8
AI for Everyone
Coursera
Python
Beginner
AI
20
4.7
TensorFlow for Beginners
Stepik
Python
Beginner
AI
30
4.6
Data Science with Python
Codecademy
Python
Beginner
AI
15
4.5
Python for Beginners
FreeCodeCamp
Python
Beginner
AI
0
4.4
Conclusion:
This project demonstrates how machine learning techniques can be applied to recommend personalized educational content. It is a practical portfolio project showcasing data processing, feature encoding, similarity calculations, and building an interactive web interface.